# Module 15 — Testing

You have been running `pytest` since module 00 and reading the test files since
module 01. This is where you write them — and where the feedback in this course
changes shape. Until now an exercise had an expected output. From here the exercise
**is** the test suite, and what gets checked is whether your tests would have caught
the bug.

`sensorlib/` next to this notebook is the code under test. `mutants/` holds three
copies of it, each with one thing changed.

One line of setup first: the notebook has to be able to import `sensorlib`, which
sits next to it. This is module 10's `sys.path` line, and module 10 said what
replaces it in a real project.

In [ ]:
import sys
from pathlib import Path

# The notebook has no __file__, so it works out where it is from the kernel's
# directory -- this folder if you opened the notebook here, the top of the
# repository if you started Jupyter there.
HERE = Path.cwd() if (Path.cwd() / "sensorlib").is_dir() else Path.cwd() / "15_testing"
sys.path.insert(0, str(HERE))  # so that `import sensorlib` finds it -- module 10

print("sensorlib is importable from", HERE.name)

## 1. `assert`, and why there is no `assertEquals`

JUnit needs `assertEquals(expected, actual)` for two reasons. A bare `assert` in Java
carries no information about what was compared — and Java assertions are **off**
unless the JVM was started with `-ea`, so no test suite can rely on them at all.

Python's `assert` has the same problem — on its own.

In [ ]:
try:
    assert 0.1 + 0.2 == 0.3
except AssertionError as err:
    print("message:", repr(str(err)), "-- empty")

`pytest` fixes it by **rewriting the assertion** when it imports your test file. It
takes the expression apart and reports the pieces. So the plain `assert` becomes the
only assertion you need, and there is no family of `assertEquals`, `assertTrue`,
`assertIn` to remember.

What that looks like in practice — these are real outputs, not paraphrases:

```
    def test_mean():
>       assert mean([1.0, 2.0]) == 2.0
E       assert 1.5 == 2.0
E        +  where 1.5 = mean([1.0, 2.0])
```

It shows the value **and** where it came from. On a list it finds the index:

```
E       assert [21.7, 91.0, 23.1] == [21.7, 91.0, 23.2]
E         At index 2 diff: 23.1 != 23.2
```

On a dict it shows only what differs:

```
E       AssertionError: assert {'tag': 'TH-04', 'unit': 'C'} == {'tag': 'TH-04', 'unit': 'F'}
E         Omitting 1 identical items, use -vv to show
E         Differing items:
E         {'unit': 'C'} != {'unit': 'F'}
```

That last one is the argument for the whole approach: with `assertEquals` you get
"expected X but was Y" and read two long lines yourself.

## 2. A test is a function whose name starts with `test_`

No class to extend, no annotation, no registration. `pytest` collects files named
`test_*.py`, functions named `test_*`, and runs them.

In [ ]:
from sensorlib.parsing import mean, parse_line


def test_parse_line_splits_tag_and_value():
    assert parse_line("TH-04;91.0") == ("TH-04", 91.0)


def test_mean_of_three():
    assert mean([1.0, 2.0, 3.0]) == 2.0


# In a notebook we call them ourselves; pytest does this for you.
test_parse_line_splits_tag_and_value()
test_mean_of_three()
print("both passed -- silence is what a passing test looks like")

**The name is the documentation.** `test_parse_line_rejects_an_empty_tag` tells a
reader of the failure report what was expected, without opening the file. That
matters more than it sounds: when a suite goes red in CI, the first thing anybody
sees is the list of names.

`test_parsing` and `test_1` are both legal and both waste the one line you get for
free.

## 3. Floats

Module 02 established that `0.1 + 0.2 != 0.3`. A test that compares floats with `==`
is therefore wrong, and it is wrong intermittently — which is worse than wrong
always.

In [ ]:
import pytest

print(0.1 + 0.2 == 0.3)
print(0.1 + 0.2 == pytest.approx(0.3))

print(mean([0.1, 0.2]) == pytest.approx(0.15))
print(0.1 + 0.2 == pytest.approx(0.3, abs=1e-12))  # the tolerance, when you need to say

`pytest.approx` works on the right-hand side of `==`, and also on lists and dicts of
floats. The default tolerance is relative, which is what you want for measured
values; `abs=` and `rel=` are there for when it is not.

The rule: **any test that compares a computed float uses `approx`.** Integers,
strings, tuples and exact-valued floats (`2.0` out of `mean([1.0, 3.0])`) do not need
it.

## 4. Testing that something raises

`pytest.raises` is a context manager — module 09's protocol, in the place you will
use it most. The test passes when the block raises that class, and fails when it does
not.

In [ ]:
from sensorlib.parsing import ParseError


def test_rejects_an_unreadable_value():
    with pytest.raises(ParseError):
        parse_line("TH-04;n/a")


def test_the_message_names_the_field():
    with pytest.raises(ParseError) as info:
        parse_line("TH-04;n/a")
    assert "n/a" in str(info.value)  # info.value is the exception object itself


test_rejects_an_unreadable_value()
test_the_message_names_the_field()
print("both passed")

Two things worth doing deliberately:

- **Put one line in the block.** Everything inside `with pytest.raises(...)` is
  covered, so a block with three lines passes when any of them raises — including a
  typo in your own test.
- **Check something about the message** when the message is part of the contract. Not
  the whole string, which makes the test break on rewording, but the piece a user
  needs: the field that was wrong, the value that was rejected.

`pytest.raises(ParseError, match="not a reading")` does the same thing in one line —
`match` is a regular expression against the message.

## 5. The same test over a table

Copying a test four times to change one number is how suites become unmaintainable.
`@pytest.mark.parametrize` — a decorator, module 14 — runs the function once per row
and reports each row separately.

In [ ]:
@pytest.mark.parametrize(
    ("line", "expected"),
    [
        ("TH-04;91.0", ("TH-04", 91.0)),
        ("  TH-04;91.0\n", ("TH-04", 91.0)),  # whitespace is stripped
        ("TH-04;-20", ("TH-04", -20.0)),  # negatives, and an int-looking value
    ],
)
def test_parse_line(line, expected):
    assert parse_line(line) == expected


for case in [("TH-04;91.0", ("TH-04", 91.0)), ("TH-04;-20", ("TH-04", -20.0))]:
    test_parse_line(*case)  # again: pytest would do this
print("passed")

Each row becomes its own test, with its own name in the report
(`test_parse_line[TH-04;-20-expected2]`), so one failing row does not hide the
others. That is the difference from a `for` loop inside one test, where the first
failure ends the test and you never learn about the rest.

The judgement call: a table is right when the rows are the **same claim** with
different data. When one row is really a different claim — "and an empty tag is
rejected" — it belongs in its own test with its own name.

## 6. Fixtures

A fixture is a function that produces something a test needs. A test asks for it by
**naming it as a parameter**, and `pytest` calls it — which is dependency injection
with no configuration.

In [ ]:
@pytest.fixture
def log():
    print("   [setup]")
    yield [21.7, 91.0]  # a generator, module 13: everything after yield is teardown
    print("   [teardown]")


def test_uses_it(log):
    assert len(log) == 2


def test_uses_it_too(log):
    assert max(log) == 91.0


print("these two need pytest -s to show anything -- they are here to be read")

Run those with `pytest -s` — without it the prints are captured and you see two dots
and nothing else — and the output is:

```
   [setup]
.   [teardown]
   [setup]
.   [teardown]
```

**Once per test, not once per file.** That is the default (`scope="function"`) and it
is the right one: each test gets a fresh object, so no test can be affected by what
another one did to it. `scope="module"` and `scope="session"` exist for expensive
setup, and they trade that isolation away.

The `yield` is module 13 and module 09 arriving together: everything before it is
setup, everything after is teardown, and the teardown runs even when the test fails.

`pytest` ships fixtures of its own. Two you will use constantly:

In [ ]:
# tmp_path: a fresh empty directory per test. pytest keeps the last three runs
# under /tmp/pytest-of-<user>/, so you can go and look at what a failing test wrote.
def test_writes_a_file(tmp_path):
    target = tmp_path / "report.txt"
    target.write_text("TH-04;91.0\n", encoding="utf-8")
    assert target.read_text(encoding="utf-8").startswith("TH-04")


# capsys: what the code under test printed.
def test_prints(capsys):
    print("hello")
    assert capsys.readouterr().out == "hello\n"


print("these two need pytest to run -- they are here to be read")

`tmp_path` is the answer to "how do I test something that writes a file" — module 08
without the temporary directory you had to make yourself. `monkeypatch` is the third
one worth knowing: it replaces an attribute for the duration of one test and puts it
back afterwards.

## 7. What makes a test worth having

A suite that always passes is worth nothing, and it is easy to write by accident.
Four properties, in the order they get violated:

**It can fail.** A test that would pass against broken code tests nothing. This is
the whole of exercise 09: your suite has to catch three planted bugs, and the check
tells you which ones got past.

**It fails for one reason.** One claim per test, and a name that says which. A test
called `test_parsing` that asserts nine things fails as one line and you learn
nothing about which of the nine broke.

**It does not depend on another test.** No shared mutable state, no ordering. Tests
run in file order today, spread across workers under `pytest -n auto` tomorrow, and
shuffled under `pytest-randomly`.

**It tests the contract, not the implementation.** Assert what the function promises
its caller. A test that reaches into `_internal` or asserts the exact wording of a
message breaks when you tidy the code up — and then it gets deleted rather than
fixed, which costs you the coverage you thought you had.

And the case worth naming separately, because it is where most of the value is:

**Test the edges.** Empty, one, the boundary, and the thing that is not a number.
`mean([])`, `readings_above(pairs, limit)` at exactly the limit, `parse_line(";91.0")`
with an empty tag, `parse_line("TH-04")` with a field missing. All three bugs in
`mutants/` live exactly there — `>` against `>=`, an empty tag, and an empty sequence
— because that is where bugs live.

## 8. Running it

```console
uv run pytest                       # everything in testpaths
uv run pytest 15_testing            # one module
uv run pytest -k parse              # only tests whose name contains "parse"
uv run pytest -x                    # stop at the first failure
uv run pytest -q                    # one character per test
uv run pytest -v                    # one line per test, with the name
uv run pytest --lf                  # only what failed last time
uv run pytest -s                    # do not swallow print output
```

`-k` and `--lf` are the two that change how a long suite feels: fix one thing, run
`--lf`, and you are looking at a second of output instead of a minute.

`-m "not your_turn"` is the marker this course uses, and `@pytest.mark.your_turn` is
declared in `pyproject.toml` — an undeclared mark is a warning, which is how a typo
in a marker name gets noticed.

---

`exercises/` is next: `exercise_01.py` to `exercise_06.py` in the usual shape, and
then **exercise 09, which is different** — you write a test suite in
`exercises/suite/`, and it is checked against three deliberately broken copies of the
library.

Module 16 leaves the machine: HTTP, `requests`, and `bytes` on the wire.